# 🐊 Cattle & Buffalo Breed Recognition — Colab Training

This notebook trains the full EfficientNet‑Lite model on the 75‑breed dataset, exports an ONNX + INT8 model, and packages everything for download to your local machine.

⚠️ **Requirements**
- The project source (`src/`, `data_collection.py`, `config.py`) must be present in the notebook session (uploaded via the Files widget or mounted from Drive).
- Two PyTorch checkpoint files are needed: `efficientnet_lite2.pth` and `efficientnet_lite4.pth`. Upload them to Drive or the Colab session.
- **Dataset option** (pick ONE):
  1. **Download directly from Kaggle** (`algsoch/breed-cattle-buffalo`) — needs `kaggle.json`
  2. **Upload `archive.zip`** from your local machine (auto-extracts + organizes)
  3. **Use pre-organized data from Google Drive** (upload `data/raw/` to Drive)

---

In [ ]:
# 1️⃣  Install runtime deps
!pip install --quiet kaggle 2>/dev/null || echo 'kaggle already installed'
!apt-get update -qq && apt-get install -y -qq jq 2>/dev/null || true

# 2️⃣  Mount Google Drive (for checkpoints + optional data)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted at /content/drive')

## 📦  Dataset source — choose ONE

**Option A: Download directly from Kaggle** (`algsoch/breed-cattle-buffalo`)
- You need a `kaggle.json` file with your credentials (place in Colab Files panel or set `KAGGLE_USERNAME`/`KAGGLE_KEY` env vars).
- The notebook will run: `kaggle datasets download -d algsoch/breed-cattle-buffalo -p /content/colab_project/data/raw`

In [ ]:
# 3A️⃣  OPTION A — Download directly from Kaggle (`algsoch/breed-cattle-buffalo`)
# ---------------------------------------------------
# This downloads the dataset directly using your kaggle credentials.
# Provide credentials via ONE of:
#   1. kaggle.json uploaded to Colab Files panel (auto-detected)
#   2. Set KAGGLE_USERNAME and KAGGLE_KEY env vars (see below)
import pathlib, os, subprocess, sys, json

# --- Credentials setup ---
# Option 1: kaggle.json in Colab Files panel (/content/kaggle.json)
# Option 2: Set env vars directly (replace with your values):
# os.environ["KAGGLE_USERNAME"] = "your_username"
# os.environ["KAGGLE_KEY"] = "your_api_key"

kaggle_dir = pathlib.Path('/root/.kaggle')
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'

if not kaggle_json.exists():
    # Try auto-copy from Colab upload location
    uploaded = pathlib.Path('/content/kaggle.json')
    if uploaded.exists():
        import shutil
        shutil.copy(uploaded, kaggle_json)
        kaggle_json.chmod(0o600)
        print('✅  Copied kaggle.json from upload')
    elif "KAGGLE_USERNAME" in os.environ and "KAGGLE_KEY" in os.environ:
        # Create kaggle.json from env vars
        creds = {"username": os.environ["KAGGLE_USERNAME"],
                "key": os.environ["KAGGLE_KEY"]}
        kaggle_json.write_text(json.dumps(creds))
        kaggle_json.chmod(0o600)
        print('✅  Created kaggle.json from environment variables')
    else:
        print('⚠️  No credentials found. Set KAGGLE_USERNAME/KAGGLE_KEY or upload kaggle.json')

ROOT = pathlib.Path('/content/colab_project')
os.makedirs(ROOT / 'data' / 'raw', exist_ok=True)

if kaggle_json.exists():
    print('Downloading dataset from Kaggle: algsoch/breed-cattle-buffalo ...')
    try:
        import kaggle
        kaggle.api.authenticate()
        kaggle.api.dataset_download_files(
            'algsoch/breed-cattle-buffalo',
            path=str(ROOT / 'data' / 'raw'),
            unzip=True
        )
        print('✅  Dataset downloaded and extracted successfully.')
        DATA_SOURCE = 'kaggle_direct'
    except Exception as e:
        print('❌  Kaggle download failed:')
        print(str(e))
        print('\n💡  Make sure credentials are valid and you have accepted dataset terms at:')
        print('   https://www.kaggle.com/datasets/algsoch/breed-cattle-buffalo')
        DATA_SOURCE = None
else:
    DATA_SOURCE = None

**Option B: Upload `archive.zip` from local machine** (if you prefer manual upload)
- Your zip should contain the raw images (any folder structure; the `organize` step will sort them into `cattle/<breed>/` and `buffalo/<breed>/` by folder name).
- After upload, the notebook runs: `python data_collection.py --source organize --input <extracted> --output data/raw`

In [ ]:
# 3B️⃣  OPTION B — Upload archive.zip from local machine (if Option A skipped)
# ---------------------------------------------------
# This cell opens a file picker. Select your archive.zip and wait for upload.
from google.colab import files
print('Upload your archive.zip (containing raw images)...')
uploaded = files.upload()  # blocks until you pick a file

import pathlib, zipfile, shutil
ROOT = pathlib.Path('/content/colab_project')
os.makedirs(ROOT / 'data' / 'raw', exist_ok=True)

if uploaded:
    zip_name = list(uploaded.keys())[0]
    print(f'Uploaded: {zip_name} ({len(uploaded[zip_name])} bytes)')
    extract_dir = ROOT / 'data' / 'raw_uploaded'
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)
    
    # Extract zip
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name]), 'r') as zf:
        zf.extractall(extract_dir)
    print(f'Extracted to {extract_dir}')
    
    # Run the organize step (same as your local command)
    # python data_collection.py --source organize --input <extracted> --output data/raw
    import subprocess, sys
    result = subprocess.run([
        sys.executable, 'data_collection.py',
        '--source', 'organize',
        '--input', str(extract_dir),
        '--output', str(ROOT / 'data' / 'raw')
    ], cwd=str(ROOT), capture_output=True, text=True)
    
    if result.returncode == 0:
        print('✅  Dataset organized successfully via `data_collection.py --source organize`')
        print(result.stdout)
        DATA_SOURCE = 'upload'
    else:
        print('❌  Organize failed:')
        print(result.stderr)
        DATA_SOURCE = None
else:
    print('⏭️  No file uploaded — will try other sources.')
    DATA_SOURCE = None

**Option C: Use pre-organized data from Google Drive**
- Upload your local `data/raw/` folder to `/content/drive/MyDrive/colab_data/raw/`.
- Upload `efficientnet_lite2.pth` and `efficientnet_lite4.pth` to the same Drive folder.

In [ ]:
# 4️⃣  Prepare paths & project source
import os, pathlib, io
ROOT = pathlib.Path('/content/colab_project')
os.makedirs(ROOT, exist_ok=True)

# Copy project source from Drive (or upload zip via Files widget)
SRC_DRIVE = pathlib.Path('/content/drive/MyDrive/colab_project')
if SRC_DRIVE.exists():
    !rsync -aq '{SRC_DRIVE}/' /content/colab_project/
    print('Project synced from Drive.')
else:
    print('⚠️  Drive project folder not found — please upload or unzip it.')

# Verify required files exist
for f in ['src/train.py', 'src/export.py', 'src/verify.py', 'src/data_pipeline.py', 'src/config.py', 'data_collection.py', 'efficientnet_lite2.pth', 'efficientnet_lite4.pth']:
    ok = pathlib.Path(f).exists() or pathlib.Path(f'/content/colab_project/{f}').exists()
    if not ok:
        print(f'⚠️  Missing: {f}')
print('✅  Required files check complete.')

# If dataset wasn't organized yet (no upload, no kaggle), try Drive copy
if DATA_SOURCE is None:
    DRIVE_DATA = pathlib.Path('/content/drive/MyDrive/colab_data/raw')
    if DRIVE_DATA.exists():
        !rsync -aq '{DRIVE_DATA}/' /content/colab_project/data/raw/
        print('Using pre-organized data from Drive.')
        DATA_SOURCE = 'drive'
    else:
        print('❌  No dataset source available. Please run Option A, B, or C.')

In [ ]:
# 5️⃣  Generate train/val/test splits
!python -m src.data_pipeline --source split --data /content/colab_project/data/raw

# Verify splits
import json
for name in ['cattle_classes.json', 'buffalo_classes.json']:
    p = pathlib.Path(f'/content/colab_project/data/splits/{name}')
    if p.exists():
        print(f'{name}: {len(json.load(open(p)))} classes')
    else:
        print(f'⚠️  Missing: {p.name}')

*Training on a T4 GPU can use a larger batch size than the default 32. The notebook sets `--batch-size 128` and `--num-workers 8` for best throughput. Adjust if you see OOM.*

In [ ]:
# 6️⃣  Train the full model (3‑phase: warmup → multi‑task → QAT)
# ------------------------------------------------------------
# Typical T4 settings:
- `--batch-size 128` (reduces if OOM)
- `--num-workers 8`
- `--device cuda`
- `--phase1-epochs 5`  (warm‑up, frozen backbone)
- `--phase2-epochs 30` (multi‑task, train heads)
- `--phase3-epochs 10` (QAT, INT8 awareness)

!python -m src.train \
  --backbone lite2 \
  --batch-size 128 \
  --num-workers 8 \
  --device cuda \
  --phase1-epochs 5 \
  --phase2-epochs 30 \
  --phase3-epochs 10

*Training takes ~20–40 min on a T4 depending on batch size and data size. The progress bars and phase logs will appear in the notebook output.*

In [ ]:
# 7️⃣  Export the model
# -------------------
# Export FP32 ONNX (static batch, required for TFLite conversion)
!python -m src.export \
  --backbone lite2 \
  --mode onnx \
  --onnx-static-batch

# Optional: Export INT8 via QAT (requires the phase‑3 checkpoint)
!python -m src.export \
  --backbone lite2 \
  --mode int8

*Exported artifacts land in `/content/colab_project/outputs/export/`. They'll be zipped and saved to Drive in the next step.*

In [ ]:
# 8️⃣  Package everything for download to your local machine
!rm -f /content/colab_project/artifacts.zip
!cd /content/colab_project && zip -r artifacts.zip \
  outputs/checkpoints/ outputs/export/ outputs/metrics/ data/splits/ src/ data_collection.py config.py
!cp /content/colab_project/artifacts.zip '/content/drive/MyDrive/colab_artifacts.zip'
print('✅  Artifacts zipped and saved to Drive. Download `colab_artifacts.zip` from Google Drive to your local machine.')

## 🎉 All done!

1.  You now have a trained model (`lite2_phase3_best.pt` or `lite2_quantized.pt`),
2.  an ONNX model (`lite2_fp32.onnx`), and
3.  metrics (`lite2_metrics.json`).

### 📥 How to use the exported model on your **local** machine

```bash
# 1. Unzip the artifacts you just downloaded:
unzip colab_artifacts.zip -d .

# 2. Run a quick inference test (same as the Python snippet in the README):
.venv/bin/python -c "
import torch
from PIL import Image
from torchvision import transforms
from src.model import BreedClassifier
model = BreedClassifier(backbone='lite2')
state = torch.load('lite2_phase3_best.pt', map_location='cpu')
model.load_state_dict(state['state_dict']); model.eval()
x = transforms.Compose([transforms.Resize(260), transforms.CenterCrop(260), transforms.ToTensor()])(Image.open('sample.jpg')).unsqueeze(0)
with torch.no_grad(): out = model(x)
print('species:', out['binary'].softmax(1))
print('top cattle:', out['cattle'].softmax(1).topk(3))
"""
# 3. Or run ONNX inference (requires onnxruntime):
import onnxruntime as ort
session = ort.InferenceSession('lite2_fp32.onnx')
img = ...  # load 260×260 float32 NCHW image
logits = session.run(None, {'input': img})[0]
```

---